In [2]:
import sys

sys.path.append('../../')

In [14]:
from ml_factory.datasets.precompute_tokens import precompute_tokens
from ml_factory.utils import merge_parts_to_dir
from ml_factory import DATA_RAW_DIR, DATA_PROCESSED_DIR
from transformers import AutoTokenizer
import torch
from pathlib import Path

In [4]:
from ml_factory.datasets import PromptBERTDataset
from torch.utils.data import Subset, DataLoader
from ml_factory.datasets.sampler import SplitSampler

In [12]:
import numpy as np
import torch

from torch.utils.data import Subset, DataLoader
from transformers import AutoModelForSequenceClassification
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [5]:
output_path = DATA_PROCESSED_DIR / '32_128_processed'

In [6]:
dataset = PromptBERTDataset(output_path)

In [7]:
sampler_data = torch.load(
    DATA_PROCESSED_DIR / "splitsampler_train_7_test_15.pt",
    weights_only=False
)

print(type(sampler_data["split"]))
print(sampler_data["split"].keys())

<class 'dict'>
dict_keys(['train', 'validation', 'test'])


In [8]:
import torch
from torch.utils.data import Subset, DataLoader

# Load previously saved split
sampler_data = torch.load(
    DATA_PROCESSED_DIR / "splitsampler_train_7_test_15.pt",
    weights_only=False
)

# Get the actual IDs
train_ids = sampler_data["split"]["train"]
val_ids = sampler_data["split"]["validation"]
test_ids = sampler_data["split"]["test"]

# Create datasets using the saved IDs
train_dataset = Subset(
    dataset,
    dataset.ids_to_position(train_ids)
)

val_dataset = Subset(
    dataset,
    dataset.ids_to_position(val_ids)
)

test_dataset = Subset(
    dataset,
    dataset.ids_to_position(test_ids)
)

In [9]:
train_loader = DataLoader(train_dataset, shuffle=True, batch_size=100)
val_loader = DataLoader(val_dataset, shuffle=False, batch_size=100)
test_loader = DataLoader(test_dataset, shuffle=False, batch_size=100)

In [10]:
import numpy as np
from torch.utils.data import Subset, DataLoader

n = 5
batch_size = 100
seed = 42

# Original 70% training prompt IDs
train_ids = np.array(sampler_data["split"]["train"])

# Shuffle and divide prompts among 5 BERTs
rng = np.random.default_rng(seed)
rng.shuffle(train_ids)

train_id_subsets = np.array_split(train_ids, n)

# Convert prompt IDs -> all corresponding chunk positions
train_subsets = [
    Subset(dataset, dataset.ids_to_position(ids.tolist()))
    for ids in train_id_subsets
]

# DataLoaders
ensemble_train_loaders = [
    DataLoader(subset, batch_size=batch_size, shuffle=True)
    for subset in train_subsets
]

In [11]:
for i, subset in enumerate(train_subsets):
    print(f"BERT {i+1}: {len(train_id_subsets[i])} prompts -> {len(subset)} chunks")

BERT 1: 50723 prompts -> 58473 chunks
BERT 2: 50722 prompts -> 58484 chunks
BERT 3: 50722 prompts -> 58454 chunks
BERT 4: 50722 prompts -> 58707 chunks
BERT 5: 50722 prompts -> 58477 chunks


In [13]:
# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "google/bert_uncased_L-2_H-128_A-2"

num_models = 5
num_epochs = 5

learning_rate = 2e-5
weight_decay = 0.01

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# ============================================================
# TRAIN 5 INDEPENDENT BERT-TINY MODELS
# ============================================================

ensemble_models = []
validation_results = []


for model_idx, train_loader in enumerate(ensemble_train_loaders):

    print("\n" + "=" * 70)
    print(f"TRAINING BERT {model_idx + 1}/{num_models}")
    print("=" * 70)


    # --------------------------------------------------------
    # Create a fresh independent BERT
    # --------------------------------------------------------

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    ).to(device)


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )


    # --------------------------------------------------------
    # Best checkpoint
    # --------------------------------------------------------

    best_val_f1 = -float("inf")
    best_model_state = None
    best_epoch = None


    # ========================================================
    # EPOCH LOOP
    # ========================================================

    for epoch in range(num_epochs):

        # ====================================================
        # TRAINING
        # ====================================================

        model.train()

        train_loss = 0.0

        progress_bar = tqdm(
            train_loader,
            desc=f"BERT {model_idx + 1} | "
                 f"Epoch {epoch + 1}/{num_epochs}"
        )


        for batch in progress_bar:

            input_ids = batch["tokenized"].to(device)
            attention_mask = batch["attention_masks"].to(device)
            labels = batch["label"].to(device)


            optimizer.zero_grad()


            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )


            loss = outputs.loss

            loss.backward()

            optimizer.step()


            train_loss += loss.item()

            progress_bar.set_postfix(
                loss=f"{loss.item():.4f}"
            )


        avg_train_loss = (
            train_loss / len(train_loader)
        )


        # ====================================================
        # VALIDATION
        #
        # IMPORTANT:
        # The SAME val_loader is used for every BERT.
        # ====================================================

        model.eval()

        val_loss = 0.0

        all_labels = []
        all_predictions = []
        all_probabilities = []


        with torch.no_grad():

            for batch in val_loader:

                input_ids = batch["tokenized"].to(device)
                attention_mask = batch["attention_masks"].to(device)
                labels = batch["label"].to(device)


                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )


                val_loss += outputs.loss.item()

                logits = outputs.logits


                probabilities = torch.softmax(
                    logits,
                    dim=1
                )[:, 1]


                predictions = torch.argmax(
                    logits,
                    dim=1
                )


                all_labels.extend(
                    labels.cpu().numpy()
                )

                all_predictions.extend(
                    predictions.cpu().numpy()
                )

                all_probabilities.extend(
                    probabilities.cpu().numpy()
                )


        # ====================================================
        # VALIDATION METRICS
        # ====================================================

        avg_val_loss = (
            val_loss / len(val_loader)
        )

        val_accuracy = accuracy_score(
            all_labels,
            all_predictions
        )

        val_precision = precision_score(
            all_labels,
            all_predictions,
            zero_division=0
        )

        val_recall = recall_score(
            all_labels,
            all_predictions,
            zero_division=0
        )

        val_f1 = f1_score(
            all_labels,
            all_predictions,
            zero_division=0
        )

        val_roc_auc = roc_auc_score(
            all_labels,
            all_probabilities
        )

        val_pr_auc = average_precision_score(
            all_labels,
            all_probabilities
        )


        # ====================================================
        # PRINT RESULTS
        # ====================================================

        print(
            f"Epoch {epoch + 1}/{num_epochs} | "
            f"Train Loss: {avg_train_loss:.4f} | "
            f"Val Loss: {avg_val_loss:.4f} | "
            f"Accuracy: {val_accuracy:.4f} | "
            f"Precision: {val_precision:.4f} | "
            f"Recall: {val_recall:.4f} | "
            f"F1: {val_f1:.4f} | "
            f"ROC-AUC: {val_roc_auc:.4f} | "
            f"PR-AUC: {val_pr_auc:.4f}"
        )


        # ====================================================
        # SAVE BEST MODEL BASED ON VALIDATION F1
        # ====================================================

        if val_f1 > best_val_f1:

            best_val_f1 = val_f1
            best_epoch = epoch + 1

            best_model_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }


    # ========================================================
    # RESTORE BEST CHECKPOINT
    # ========================================================

    model.load_state_dict(best_model_state)
    model.to(device)


    # ========================================================
    # STORE TRAINED MODEL
    # ========================================================

    ensemble_models.append(model)


    validation_results.append({
        "model": model_idx + 1,
        "best_epoch": best_epoch,
        "best_val_f1": best_val_f1
    })


    print(
        f"\nBERT {model_idx + 1} "
        f"best epoch: {best_epoch}"
    )

    print(
        f"BERT {model_idx + 1} "
        f"best validation F1: {best_val_f1:.4f}"
    )


# ============================================================
# COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("ALL 5 BERT MODELS HAVE BEEN TRAINED")
print("=" * 70)

for result in validation_results:

    print(
        f"BERT {result['model']}: "
        f"Best Epoch = {result['best_epoch']} | "
        f"Best Val F1 = {result['best_val_f1']:.4f}"
    )

Device: cuda

TRAINING BERT 1/5


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT 1 | Epoch 1/5:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 1/5 | Train Loss: 0.3951 | Val Loss: 0.2027 | Accuracy: 0.9251 | Precision: 0.9349 | Recall: 0.9236 | F1: 0.9292 | ROC-AUC: 0.9785 | PR-AUC: 0.9827


...

Epoch 3/5 | Train Loss: 0.1530 | Val Loss: 0.1305 | Accuracy: 0.9497 | Precision: 0.9579 | Recall: 0.9471 | F1: 0.9524 | ROC-AUC: 0.9904 | PR-AUC: 0.9921


BERT 5 | Epoch 4/5:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 4/5 | Train Loss: 0.1302 | Val Loss: 0.1188 | Accuracy: 0.9542 | Precision: 0.9543 | Recall: 0.9600 | F1: 0.9572 | ROC-AUC: 0.9919 | PR-AUC: 0.9933


BERT 5 | Epoch 5/5:   0%|          | 0/585 [00:00<?, ?it/s]

Epoch 5/5 | Train Loss: 0.1133 | Val Loss: 0.1111 | Accuracy: 0.9566 | Precision: 0.9636 | Recall: 0.9546 | F1: 0.9591 | ROC-AUC: 0.9928 | PR-AUC: 0.9941

BERT 5 best epoch: 5
BERT 5 best validation F1: 0.9591

ALL 5 BERT MODELS HAVE BEEN TRAINED
BERT 1: Best Epoch = 5 | Best Val F1 = 0.9597
BERT 2: Best Epoch = 5 | Best Val F1 = 0.9596
BERT 3: Best Epoch = 5 | Best Val F1 = 0.9567
BERT 4: Best Epoch = 5 | Best Val F1 = 0.9597
BERT 5: Best Epoch = 5 | Best Val F1 = 0.9591


## Saving best models

In [21]:
print("Number of models in ensemble_models:", len(ensemble_models))
print("Number of validation results:", len(validation_results))

Number of models in ensemble_models: 7
Number of validation results: 5


In [22]:
print("Total models:", len(ensemble_models))

for i, model in enumerate(ensemble_models):
    print(i, id(model))

Total models: 7
0 2986095683024
1 3004823341536
2 3004823353152
3 3004823350272
4 2983079530368
5 2983079530368
6 2983079530368


In [18]:
save_dir = DATA_PROCESSED_DIR / "ensemble_models"

save_dir.mkdir(parents=True, exist_ok=True)

In [23]:
# ============================================================
# SAVE THE 5 MODELS FROM THE LATEST TRAINING RUN
# ============================================================

latest_models = ensemble_models[-5:]

print("Saving", len(latest_models), "models")

for i, model in enumerate(latest_models, start=1):

    result = validation_results[i - 1]

    model_path = (
        save_dir /
        f"bert_tiny_ensemble_model_{i}.pt"
    )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "model_name": MODEL_NAME,
            "model_index": i,
            "best_epoch": result["best_epoch"],
            "best_val_f1": result["best_val_f1"],
        },
        model_path
    )

    print(
        f"Saved Model {i} | "
        f"Best Epoch: {result['best_epoch']} | "
        f"Best F1: {result['best_val_f1']:.4f}"
    )

Saving 5 models
Saved Model 1 | Best Epoch: 5 | Best F1: 0.9597
Saved Model 2 | Best Epoch: 5 | Best F1: 0.9596
Saved Model 3 | Best Epoch: 5 | Best F1: 0.9567
Saved Model 4 | Best Epoch: 5 | Best F1: 0.9597
Saved Model 5 | Best Epoch: 5 | Best F1: 0.9591


In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loaded_models = []

for model_idx in range(1, 6):

    model_path = save_dir / f"bert_tiny_ensemble_model_{model_idx}.pt"

    checkpoint = torch.load(
        model_path,
        map_location=device,
        weights_only=False
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=2
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    loaded_models.append(model)

    print(
        f"Loaded Model {model_idx} | "
        f"Best epoch: {checkpoint['best_epoch']} | "
        f"Best val F1: {checkpoint['best_val_f1']:.4f}"
    )

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded Model 1 | Best epoch: 5 | Best val F1: 0.9597


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded Model 4 | Best epoch: 5 | Best val F1: 0.9597


Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/bert_uncased_L-2_H-128_A-2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded Model 5 | Best epoch: 5 | Best val F1: 0.9591


## Testing


In [25]:
import time
import numpy as np
import torch

all_model_probabilities = []
test_labels = None

# Start timing the complete ensemble inference
if device.type == "cuda":
    torch.cuda.synchronize()

start_time = time.perf_counter()

for model_idx, model in enumerate(loaded_models):

    model.eval()

    model_probabilities = []
    model_labels = []

    with torch.no_grad():

        for batch in test_loader:

            input_ids = batch["tokenized"].to(device)
            attention_mask = batch["attention_masks"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probabilities = torch.softmax(
                outputs.logits,
                dim=1
            )[:, 1]

            model_probabilities.extend(
                probabilities.cpu().numpy()
            )

            model_labels.extend(
                labels.cpu().numpy()
            )

    all_model_probabilities.append(
        np.array(model_probabilities)
    )

    if test_labels is None:
        test_labels = np.array(model_labels)

# Make sure all GPU operations have finished before stopping timer
if device.type == "cuda":
    torch.cuda.synchronize()

end_time = time.perf_counter()

total_inference_time = end_time - start_time

print(f"Total ensemble inference time: {total_inference_time:.4f} seconds")

Total ensemble inference time: 92.8784 seconds


## per-example latency

In [26]:
num_test_examples = len(test_labels)

latency_per_example = (
    total_inference_time / num_test_examples
)

print(f"Number of test examples: {num_test_examples}")
print(f"Latency per example: {latency_per_example:.6f} seconds")
print(
    f"Latency per example: "
    f"{latency_per_example * 1000:.3f} ms"
)

Number of test examples: 63139
Latency per example: 0.001471 seconds
Latency per example: 1.471 ms


In [27]:
print([len(x) for x in all_model_probabilities])
print(len(test_labels))

[63139, 63139, 63139, 63139, 63139]
63139


In [28]:
model_probabilities_matrix = np.vstack(all_model_probabilities)

print(model_probabilities_matrix.shape)

(5, 63139)


In [29]:
ensemble_probabilities = np.mean(
    model_probabilities_matrix,
    axis=0
)

print(ensemble_probabilities.shape)

(63139,)


In [30]:
ensemble_predictions = (
    ensemble_probabilities >= 0.5
).astype(int)

print(ensemble_predictions[:20])

[1 0 0 1 0 0 1 1 0 0 0 1 0 0 0 0 1 0 1 1]


In [31]:
ensemble_accuracy = accuracy_score(
    test_labels,
    ensemble_predictions
)

ensemble_precision = precision_score(
    test_labels,
    ensemble_predictions,
    zero_division=0
)

ensemble_recall = recall_score(
    test_labels,
    ensemble_predictions,
    zero_division=0
)

ensemble_f1 = f1_score(
    test_labels,
    ensemble_predictions,
    zero_division=0
)

ensemble_roc_auc = roc_auc_score(
    test_labels,
    ensemble_probabilities
)

ensemble_pr_auc = average_precision_score(
    test_labels,
    ensemble_probabilities
)

In [32]:
print("Ensemble Test Results")
print("---------------------")
print(f"Accuracy : {ensemble_accuracy:.4f}")
print(f"Precision: {ensemble_precision:.4f}")
print(f"Recall   : {ensemble_recall:.4f}")
print(f"F1       : {ensemble_f1:.4f}")
print(f"ROC-AUC  : {ensemble_roc_auc:.4f}")
print(f"PR-AUC   : {ensemble_pr_auc:.4f}")

Ensemble Test Results
---------------------
Accuracy : 0.9581
Precision: 0.9618
Recall   : 0.9599
F1       : 0.9609
ROC-AUC  : 0.9932
PR-AUC   : 0.9945
